In [1]:
from stockfish import Stockfish
import chess.pgn
import chess.engine
import io
import matplotlib.pyplot as plt


## Variables for user to change


In [2]:
STOCKFISH_PATH  = "C:\\Users\\user\\PycharmProjects\\Databases\\stockfish-windows-x86-64-avx2\\stockfish\\stockfish-windows-x86-64-avx2.exe"
DEPTH = 20

In [3]:
game_pgn = """
[Event "Live Chess"]
[Site "Chess.com"]
[Date "2026.07.06"]
[Round "-"]
[White "SahilNubFtw"]
[Black "rajakel"]
[Result "0-1"]
[CurrentPosition "8/ppk2p1p/2p1p1p1/8/PPP1P2P/6P1/5br1/1R2K3 w - - 1 30"]
[Timezone "UTC"]
[ECO "B01"]
[ECOUrl "https://www.chess.com/openings/Scandinavian-Defense-Mieses-Kotrc-Main-Line"]
[UTCDate "2026.07.06"]
[UTCTime "07:49:47"]
[WhiteElo "744"]
[BlackElo "770"]
[TimeControl "180"]
[Termination "rajakel won by resignation"]
[StartTime "07:49:47"]
[EndDate "2026.07.06"]
[EndTime "07:49:47"]
[Link "https://www.chess.com/analysis/game/live/171186436586/analysis?move=57"]
[WhiteUrl "https://images.chesscomfiles.com/uploads/v1/user/183555809.4b7f9c5f.50x50o.79241cd076bc.jpg"]
[WhiteCountry "69"]
[WhiteTitle ""]
[BlackUrl "https://www.chess.com/bundles/web/images/noavatar_l.84a92436.gif"]
[BlackCountry "86"]
[BlackTitle ""]

1. e4 d5 2. exd5 Qxd5 3. Nc3 Qa5 4. d3 c6 5. Nf3 Bf5 6. g3 e6 7. Bg2 Nd7 8. O-O
O-O-O 9. Bd2 Ngf6 10. Ne4 Qc7 11. Nxf6 Nxf6 12. b3 g6 13. Ne1 Bg7 14. c4 Ng4 15.
Rb1 Bxd3 16. Bf4 Bxf1 17. Bxc7 Rxd1 18. Rxd1 Bxg2 19. Nxg2 Kxc7 20. Ne3 Nxe3 21.
fxe3 Rd8 22. Re1 Rd2 23. a4 Bc3 24. Rb1 Ra2 25. h4 Bd2 26. e4 Be3+ 27. Kf1 Rf2+
28. Ke1 Rg2 29. b4 Bf2+ 0-1
"""

In [4]:
#optional, in case pasted directly from chess.com, to extract only the pgn
game_pgn = "1." + game_pgn.split("\n1.")[1]
game_pgn

'1. e4 d5 2. exd5 Qxd5 3. Nc3 Qa5 4. d3 c6 5. Nf3 Bf5 6. g3 e6 7. Bg2 Nd7 8. O-O\nO-O-O 9. Bd2 Ngf6 10. Ne4 Qc7 11. Nxf6 Nxf6 12. b3 g6 13. Ne1 Bg7 14. c4 Ng4 15.\nRb1 Bxd3 16. Bf4 Bxf1 17. Bxc7 Rxd1 18. Rxd1 Bxg2 19. Nxg2 Kxc7 20. Ne3 Nxe3 21.\nfxe3 Rd8 22. Re1 Rd2 23. a4 Bc3 24. Rb1 Ra2 25. h4 Bd2 26. e4 Be3+ 27. Kf1 Rf2+\n28. Ke1 Rg2 29. b4 Bf2+ 0-1\n'

In [5]:
def plot_eval_graph(scores, clip=800, markers=None):
    """
    scores: list of chess.engine.Cp or Mate scores
    markers: optional list of tuples: (index, color)
             example: [(12, "#f04f3e"), (13, "#5b9bd5")]
    """

    y = []
    mates = []

    for s in scores:
        if s.is_mate():
            m = s.mate()
            y.append(clip if m > 0 else -clip)
            mates.append((len(y) - 1, m))
        else:
            cp = s.score()
            y.append(max(-clip, min(clip, cp)))

    x = list(range(len(y)))

    white_y = []
    black_y = []
    black_background = []
    for value in y:
        white_y.append(min(value, 0))
        black_y.append(max(value, 0))
        black_background.append(-clip)

    fig, ax = plt.subplots(figsize=(14, 2.5))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    ax.step(x, y, where="mid", linewidth=1.3)
    ax.step(x, white_y, where="mid", linewidth=1.3)
    ax.step(x, black_y, where="mid", linewidth=1.3)

    # Chess.com-like filled evaluation silhouette
    ax.fill_between(
        x, black_background, 0,
        color="#000000",
        alpha=1.0,
        linewidth=0,
        zorder=1,
    )
    # Subtle outline on top of the filled area
    ax.plot(
        x, black_background,
        color="#000000",
        linewidth=0.0,
        zorder=1,
    )

    ax.fill_between(
        x, white_y, 0,
        color="#ffffff",
        alpha=1.0,
        linewidth=0,
        zorder=1,
    )
    # Subtle outline on top of the filled area
    ax.plot(
        x, white_y,
        color="#cccccc",
        linewidth=1.3,
        zorder=2,
    )


    ax.fill_between(
        x, black_y, 0,
        color="#000000",
        alpha=1.0,
        linewidth=0,
        zorder=1,
    )


    # Subtle outline on top of the filled area
    ax.plot(
        x, black_y,
        color="#333333",
        linewidth=0.0,
        zorder=2,
    )

     # Equality line
    ax.axhline(
        0,
        color="#888888",
        linewidth=2.0,
        zorder=3,
    )

    # Optional colored move markers
    if markers:
        for idx, color in markers:
            if 0 <= idx < len(y):
                ax.scatter(
                    idx, y[idx],
                    s=85,
                    color=color,
                    edgecolor="white",
                    linewidth=1.2,
                    zorder=4,
                )

    # Mate labels
    for idx, mate in mates:
        ax.text(
            idx,
            y[idx],
            f"M{abs(mate)}",
            ha="center",
            va="bottom" if mate > 0 else "top",
            fontsize=8,
            color="white",
            zorder=5,
        )

    # Match Chess.com: no axis labels, ticks, or gridlines
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.set_xlim(0, max(len(x) - 1, 1))
    ax.set_ylim(-clip, clip)

    # Rounded-looking clean frame approximation
    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.margins(x=0, y=0)
    plt.tight_layout(pad=0)
    plt.show()

In [6]:
engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
engine.configure({
    "Threads": 16,          # set to your CPU cores
    "Hash": 8192,          # MB RAM for Stockfish
})

In [7]:
#wrap string using io library so the chess library can read it as a "file" (thanks chatGPT)
game = chess.pgn.read_game(io.StringIO(game_pgn))

In [8]:
board = game.board()
rows = []
eval_bar = []

In [9]:
for ply, move in enumerate(game.mainline_moves(), start=1):
    board.push(move)

    info = engine.analyse(
        board,
        chess.engine.Limit(depth=DEPTH),
    )

    print(
        ply,
        move.uci(),
        info["score"].white(),
    )
    eval_bar.append(info["score"].black())
engine.quit()

1 e2e4 +33
2 d7d5 +70
3 e4d5 +68
4 d8d5 +67
5 b1c3 +71
6 d5a5 +71
7 d2d3 +24
8 c7c6 +55
9 g1f3 +54
10 c8f5 +52
11 g2g3 +50
12 e7e6 +59
13 f1g2 +53
14 b8d7 +51
15 e1g1 +45
16 e8c8 +181
17 c1d2 +202
18 g8f6 +207
19 c3e4 +157
20 a5c7 +157
21 e4f6 +163
22 d7f6 +166
23 b2b3 +79
24 g7g6 +183
25 f3e1 +29
26 f8g7 +39
27 c2c4 -110
28 f6g4 -42
29 a1b1 -103
30 f5d3 -8
31 d2f4 -326
32 d3f1 -259
33 f4c7 -498
34 d8d1 -498
35 b1d1 -507
36 f1g2 -502
37 e1g2 -531
38 c8c7 -536
39 g2e3 -631
40 g4e3 -666
41 f2e3 -652


KeyboardInterrupt: 

In [ ]:
plot_eval_graph(eval_bar)